[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/BMED365-2026/blob/main/Lab3-GenAI-LLM/notebooks/09-chatgpt-claude-api.ipynb)

**Lab3-GenAI-LLM: 09-chatgpt-claude-api.ipynb** (BMED365) | Priority: 2 (supplementary)

_Practical integration of ChatGPT and Claude APIs for medical applications – AI-assisted notebook_

Last updated: 2026-01-19, A. Lundervold

Use the `bmed365-2026` conda environment or Google Colab

---

> **Note**: This notebook is optional and requires API keys from OpenAI and/or Anthropic. It is intended for students who want to experiment with programmatic use of LLMs. Without API keys, the notebook will demonstrate concepts using simulated responses.

## Learning Objectives

By the end of this notebook, you will be able to:

| Objective | Description |
|-----------|-------------|
| **Configure API access** | Set up and securely manage OpenAI and Anthropic API keys using environment variables |
| **Make API calls** | Implement functions to communicate with ChatGPT and Claude models programmatically |
| **Handle errors robustly** | Implement retry logic, rate limiting, and graceful error handling |
| **Build medical assistants** | Create LLM-based functions with appropriate safety constraints for healthcare contexts |
| **Understand token economics** | Track and calculate API usage costs based on input/output tokens |
| **Apply security best practices** | Implement logging, privacy protection, and human oversight in medical AI systems |
| **Compare model capabilities** | Understand the differences between OpenAI and Anthropic APIs and when to use each |

### Prerequisites

This notebook assumes familiarity with:
- Python programming basics
- LLM concepts from notebooks 01-03 in this lab
- Prompt engineering principles from notebook 04

## Contents

| Section | Topic | Key Skills |
|---------|-------|------------|
| 1 | [Setup and Configuration](#1-setup-and-configuration) | API keys, environment variables, security |
| 2 | [OpenAI API (ChatGPT)](#2-openai-api-chatgpt) | Making API calls, temperature, system prompts |
| 3 | [Anthropic API (Claude)](#3-anthropic-api-claude) | Token usage, cost calculation, model selection |
| 4 | [Error Handling](#4-error-handling) | Retry logic, rate limiting, robust clients |
| 5 | [Practical Examples](#5-practical-examples) | Medical assistant, safety instructions |
| 6 | [Security Considerations](#6-security-considerations) | Privacy, hallucination, logging, oversight |
| 7 | [Summary & Key Takeaways](#summary) | Core concepts recap |
| 8 | [Glossary](#glossary-of-key-terms) | Key terminology |
| 9 | [Further Reading](#further-reading--resources) | Resources for deeper learning |

---

## Reproducibility Note

This notebook uses external APIs which may have non-deterministic behavior. To maximize reproducibility:
- Use `temperature=0` for deterministic outputs (when supported)
- Pin specific model versions (e.g., `gpt-4-0125-preview` instead of `gpt-4`)
- Log all API responses for traceability
- Note that model behavior may change over time as providers update their systems

---

## 1. Setup and configuration

### Prerequisites

1. API key from [OpenAI](https://platform.openai.com/api-keys) and/or [Anthropic](https://console.anthropic.com/)
2. Python packages: `openai`, `anthropic`

### Best practices for API keys

**Never** hardcode API keys in code. Use environment variables:

```bash
# In terminal:
export OPENAI_API_KEY="sk-..."
export ANTHROPIC_API_KEY="sk-ant-..."
```

Or a `.env` file (which should NOT be committed to git).

### How to set up API keys

If you see "Not configured" for either API, follow these steps:

#### Getting your API keys

1. **OpenAI (ChatGPT)**: 
   - Visit [OpenAI API Keys](https://platform.openai.com/api-keys)
   - Sign up or log in
   - Create a new API key
   - Copy the key (starts with `sk-`)

2. **Anthropic (Claude)**:
   - Visit [Anthropic Console](https://console.anthropic.com/)
   - Sign up or log in
   - Navigate to "API Keys"
   - Create a new key
   - Copy the key (starts with `sk-ant-`)

#### Setting up the keys (choose one method)

**Method 1: Using a `.env` file (Recommended)**

1. Create a `.env` file in the same directory as this notebook (or in the project root)
2. Add your keys:
   ```bash
   OPENAI_API_KEY=sk-your-key-here
   ANTHROPIC_API_KEY=sk-ant-your-key-here
   ```
3. **Important**: Make sure `.env` is in your `.gitignore` file (never commit API keys!)
4. Re-run the setup cell above - the notebook will automatically load keys from `.env`

**Method 2: Terminal environment variables**

```bash
export OPENAI_API_KEY="sk-your-key-here"
export ANTHROPIC_API_KEY="sk-ant-your-key-here"
```

**Method 3: Google Colab**

If running in Colab, you can set environment variables in a code cell:
```python
import os
os.environ["OPENAI_API_KEY"] = "sk-your-key-here"
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-your-key-here"
```

After setting up your keys, **re-run the setup cell** above. You should see:
```
API availability:
  OpenAI (ChatGPT): Available
  Anthropic (Claude): Available
```

If keys are not configured, the notebook will use simulated responses for demonstration purposes.

In [1]:
# =============================================================================
# ENVIRONMENT SETUP AND DEPENDENCY MANAGEMENT
# =============================================================================
# This cell handles:
#   1. Package installation (if running in Colab)
#   2. Environment variable loading (for secure API key management)
#   3. API availability checking
#
# Required packages:
#   - openai>=1.0.0    : Official OpenAI Python client
#   - anthropic>=0.18  : Official Anthropic Python client  
#   - python-dotenv    : For loading .env files
# =============================================================================

# Uncomment to install packages locally:
# !pip install openai anthropic python-dotenv

import os
import sys
from datetime import datetime

# -------------------------
# Step 1: Environment Detection
# -------------------------
# Detect if we're running in Google Colab (affects installation behavior)
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🌐 Running in Google Colab")
    print("   Installing required packages...")
    !pip install openai anthropic python-dotenv --quiet
    print("   ✓ Packages installed")

# -------------------------
# Step 2: Load Environment Variables
# -------------------------
# Best practice: Store API keys in .env file (never commit to git!)
# The .env file should contain:
#   OPENAI_API_KEY=sk-...
#   ANTHROPIC_API_KEY=sk-ant-...

try:
    from dotenv import load_dotenv
    # load_dotenv() searches for .env in current directory and parent directories
    load_dotenv()
    print("✓ Environment variables loaded from .env file")
except ImportError:
    print("⚠ python-dotenv not installed - using existing environment variables only")

# -------------------------
# Step 3: API Availability Check
# -------------------------
# Check which APIs are configured and available
OPENAI_AVAILABLE = bool(os.getenv("OPENAI_API_KEY"))
ANTHROPIC_AVAILABLE = bool(os.getenv("ANTHROPIC_API_KEY"))

print(f"\n{'='*50}")
print(f"API Configuration Status ({datetime.now().strftime('%Y-%m-%d %H:%M')})")
print(f"{'='*50}")
print(f"  OpenAI (ChatGPT):   {'✓ Available' if OPENAI_AVAILABLE else '✗ Not configured'}")
print(f"  Anthropic (Claude): {'✓ Available' if ANTHROPIC_AVAILABLE else '✗ Not configured'}")
print(f"{'='*50}")

if not (OPENAI_AVAILABLE or ANTHROPIC_AVAILABLE):
    print("\n⚠️  No API keys found.")
    print("   → Examples will use simulated responses for demonstration.")
    print("   → See cell above for instructions on setting up API keys.")

✓ Environment variables loaded from .env file

API Configuration Status (2026-01-19 01:26)
  OpenAI (ChatGPT):   ✓ Available
  Anthropic (Claude): ✓ Available


### ✅ Check Your Understanding: API Setup

**Q1:** Why should you never hardcode API keys directly in your notebook code?

<details>
<summary>Click for answer</summary>

**Answer:** Hardcoded API keys can be accidentally committed to version control (like GitHub), exposing them publicly. This can lead to:
- **Unauthorized usage** and unexpected billing charges
- **Security breaches** if the key provides access to sensitive systems
- **Key revocation** by the provider, breaking your applications

Best practice: Use environment variables or `.env` files that are excluded from git via `.gitignore`.
</details>

**Q2:** What is the purpose of `load_dotenv()` and when is it called?

<details>
<summary>Click for answer</summary>

**Answer:** `load_dotenv()` reads a `.env` file and loads its contents as environment variables. This allows you to:
- Keep sensitive keys separate from code
- Use the same code on different machines with different configurations
- Easily switch between development and production API keys

It should be called early in your script, before any code that needs the environment variables.
</details>

**Q3:** What happens in this notebook if no API keys are configured?

<details>
<summary>Click for answer</summary>

**Answer:** The notebook uses **simulated responses** for demonstration purposes. This allows students to understand the code structure and API patterns without needing actual API access. The simulated responses are clearly marked with `[SIMULATED RESPONSE]`.
</details>

---

## 2. OpenAI API (ChatGPT)

In [2]:
# =============================================================================
# OPENAI (ChatGPT) API WRAPPER FUNCTION
# =============================================================================
# This function provides a clean interface for making ChatGPT API calls with:
#   - Configurable model selection (gpt-3.5-turbo, gpt-4, gpt-4-turbo, etc.)
#   - Temperature control for determinism vs creativity
#   - System prompt support for role/behavior specification
#   - Graceful fallback to simulated responses when API is unavailable
#
# Key concepts:
#   - Messages format: OpenAI uses a list of message objects with "role" and "content"
#   - Roles: "system" (sets behavior), "user" (your input), "assistant" (model output)
#   - Temperature: 0 = deterministic, 1 = balanced, 2 = maximum creativity
# =============================================================================

from typing import Optional

def chat_with_gpt(prompt: str, 
                  model: str = "gpt-3.5-turbo",
                  temperature: float = 0.7,
                  system_prompt: str = None) -> str:
    """
    Send a message to ChatGPT and get a response.
    
    This is a minimal wrapper around the OpenAI Chat Completions API.
    For production use, consider adding: token counting, cost tracking,
    streaming support, and more robust error handling.
    
    Args:
        prompt (str): The user's message/question to send to the model
        model (str): Which GPT model to use. Options include:
            - "gpt-3.5-turbo": Fast, cost-effective (default)
            - "gpt-4": More capable, slower, more expensive
            - "gpt-4-turbo": Latest GPT-4 with 128k context
        temperature (float): Controls randomness (0-2):
            - 0: Deterministic, same input → same output (good for facts)
            - 0.3-0.7: Balanced (good for most tasks)
            - 1.0+: Creative, varied outputs (good for brainstorming)
        system_prompt (str, optional): Instructions that define the model's behavior
            Example: "You are a medical assistant. Be concise and accurate."
    
    Returns:
        str: The model's response text, or error message if call fails
    
    Medical AI Note:
        For clinical applications, use temperature=0 for reproducibility
        and include appropriate disclaimers in system prompts.
    """
    # Fallback for demonstration when API key is not configured
    if not OPENAI_AVAILABLE:
        return f"[SIMULATED RESPONSE]\nThis is a simulated response for: {prompt[:100]}..."
    
    try:
        # Import and initialize OpenAI client
        # Client automatically uses OPENAI_API_KEY from environment
        from openai import OpenAI
        client = OpenAI()
        
        # Build the messages list following OpenAI's chat format
        # System message (if provided) sets the assistant's behavior
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        # User message contains the actual query
        messages.append({"role": "user", "content": prompt})
        
        # Make the API call
        # This is a synchronous call - for production, consider async
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature
        )
        
        # Extract and return the response text
        # response.choices is a list (can request multiple completions)
        # We take the first (index 0) completion's message content
        return response.choices[0].message.content
        
    except Exception as e:
        # Catch-all for API errors, network issues, etc.
        # In production, handle specific exceptions differently
        return f"Error in API call: {e}"


# =============================================================================
# EXAMPLE: Medical Information Query
# =============================================================================
# Demonstrating the function with a clinical question
# Using low temperature (0.3) for factual, consistent responses

response = chat_with_gpt(
    prompt="Briefly explain what HbA1c measures.",
    temperature=0.3,  # Low temperature for factual accuracy
    system_prompt="You are a helpful medical assistant. Answer briefly and precisely."
)

print("=" * 60)
print("CHATGPT API EXAMPLE: Medical Information Query")
print("=" * 60)
print(f"\nModel: gpt-3.5-turbo | Temperature: 0.3")
print(f"\nResponse:\n{response}")

CHATGPT API EXAMPLE: Medical Information Query

Model: gpt-3.5-turbo | Temperature: 0.3

Response:
HbA1c measures the average blood sugar levels over the past 2-3 months.


### ✅ Check Your Understanding: OpenAI API

**Q1:** What is the purpose of the `system_prompt` parameter, and why is it especially important in medical AI applications?

<details>
<summary>Click for answer</summary>

**Answer:** The `system_prompt` sets the model's behavior, persona, and constraints. In medical AI, it's crucial for:
- **Setting safety boundaries** (e.g., "Always recommend consulting a healthcare professional")
- **Defining expertise level** (e.g., "Explain in patient-friendly terms")
- **Enforcing disclaimers** (e.g., "You provide general information, not medical advice")
- **Controlling scope** (e.g., "Only answer questions about diabetes management")
</details>

**Q2:** Why would you use `temperature=0` for a clinical decision support system but `temperature=0.8` for generating patient education materials?

<details>
<summary>Click for answer</summary>

**Answer:** 
- **Temperature=0** (deterministic): For clinical decision support, you want **consistent, reproducible outputs**. The same patient data should produce the same recommendations. Variability could be dangerous.
- **Temperature=0.8** (creative): For patient education, some variation in phrasing can make content more engaging and allow for different explanations if a patient doesn't understand the first one. Creativity is acceptable when accuracy requirements are less strict.
</details>

**Q3:** The function returns `response.choices[0].message.content`. Why is `choices` a list, and when might you want `choices[1]` or more?

<details>
<summary>Click for answer</summary>

**Answer:** OpenAI's API can generate multiple completions for the same prompt (using the `n` parameter). This is useful for:
- **Comparison**: Generate several responses and pick the best
- **Voting/consensus**: Have multiple responses and use majority answer
- **Diversity**: Offer users different phrasings or approaches

In most cases, we only request one completion (`n=1`), so we use `choices[0]`.
</details>

---

## 3. Anthropic API (Claude)

Claude is Anthropic's family of AI assistants, known for being helpful, harmless, and honest. The API structure differs slightly from OpenAI's.

### Key Differences: OpenAI vs Anthropic APIs

| Aspect | OpenAI (ChatGPT) | Anthropic (Claude) |
|--------|------------------|-------------------|
| **System prompt** | Part of messages array | Separate `system` parameter |
| **Temperature range** | 0-2 | 0-1 |
| **Token info** | Requires separate call | Included in response |
| **Message format** | `{"role": ..., "content": ...}` | Same structure |
| **Response structure** | `response.choices[0].message.content` | `response.content[0].text` |

### Available Models (as of January 2026)

| Model | Characteristics | Use Case |
|-------|-----------------|----------|
| **claude-sonnet-4-5-20250929** | Fast, cost-effective | General tasks, high volume |
| **claude-opus-4-5-20251101** | Most capable | Complex reasoning, analysis |
| **claude-haiku-4-5-20251001** | Fastest, cheapest | Simple tasks, real-time |

> **Medical AI Note:** Claude models are trained with a strong emphasis on safety and helpfulness. For medical applications, Claude's tendency to acknowledge uncertainty and recommend professional consultation can be advantageous.

See: https://docs.anthropic.com for full documentation

In [3]:
# =============================================================================
# ANTHROPIC (Claude) API WITH COST TRACKING
# =============================================================================
# This section demonstrates the Anthropic API with two important additions:
#   1. Token usage tracking (built into Claude's API response)
#   2. Cost calculation for budgeting and monitoring
#
# Key concept: API costs are based on TOKEN usage, not character count
#   - Tokens are roughly 4 characters or 0.75 words on average
#   - Input tokens (your prompt) and output tokens (model response) are priced differently
#   - Longer prompts/responses = higher costs
#
# Medical AI consideration: Cost tracking is essential for:
#   - Budget management in research projects
#   - Per-query cost analysis for clinical tools
#   - Optimizing prompt length for cost efficiency
# =============================================================================

# -------------------------
# PRICING CONFIGURATION (as of January 2026)
# -------------------------
# Prices are per million tokens - update these when pricing changes
# Note: Pricing may vary for prompts > 200K tokens (extended context)

CLAUDE_PRICING = {
    # Claude Sonnet 4.5: Balanced performance and cost
    "claude-sonnet-4-5-20250929": {
        "input_per_million": 3.00,    # $3.00 per 1M input tokens
        "output_per_million": 15.00   # $15.00 per 1M output tokens
    },
    # Claude Opus 4.5: Most capable, highest cost
    "claude-opus-4-5-20251101": {
        "input_per_million": 15.00,   # $15.00 per 1M input tokens
        "output_per_million": 75.00   # $75.00 per 1M output tokens
    },
    # Claude Haiku 4.5: Fastest and most affordable
    "claude-haiku-4-5-20251001": {
        "input_per_million": 0.80,    # $0.80 per 1M input tokens
        "output_per_million": 4.00    # $4.00 per 1M output tokens
    },
    # Fallback for unknown models (uses Sonnet pricing)
    "default": {
        "input_per_million": 3.00,
        "output_per_million": 15.00
    }
}


def calculate_cost(model: str, input_tokens: int, output_tokens: int) -> dict:
    """
    Calculate the cost of an API call based on model and token usage.
    
    This function is essential for:
    - Budget tracking in research projects
    - Cost optimization (choosing appropriate models)
    - Billing analysis in production systems
    
    Args:
        model (str): The model identifier returned by the API
        input_tokens (int): Number of tokens in the prompt/input
        output_tokens (int): Number of tokens in the response/output
    
    Returns:
        dict: Contains 'input_cost', 'output_cost', and 'total_cost' in USD
    
    Example:
        >>> calculate_cost("claude-sonnet-4-5-20250929", 1000, 500)
        {'input_cost': 0.003, 'output_cost': 0.0075, 'total_cost': 0.0105}
    """
    # Get pricing for the specific model, fall back to default if unknown
    pricing = CLAUDE_PRICING.get(model, CLAUDE_PRICING["default"])
    
    # Calculate costs: (tokens / 1,000,000) * price_per_million
    input_cost = (input_tokens / 1_000_000) * pricing["input_per_million"]
    output_cost = (output_tokens / 1_000_000) * pricing["output_per_million"]
    total_cost = input_cost + output_cost
    
    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


def chat_with_claude(prompt: str,
                     model: str = "claude-sonnet-4-5-20250929",
                     temperature: float = 0.7,
                     system_prompt: str = None) -> dict:
    """
    Send a message to Claude and get a response with usage statistics.
    
    Unlike the OpenAI wrapper, this function returns a dictionary with both
    the response text AND metadata including token usage and cost estimates.
    This is important for monitoring API usage in production systems.
    
    Args:
        prompt (str): The user's message/question
        model (str): Which Claude model to use:
            - "claude-sonnet-4-5-20250929": Fast, cost-effective (default)
            - "claude-opus-4-5-20251101": Most capable, expensive
            - "claude-haiku-4-5-20251001": Fastest, cheapest
        temperature (float): Controls randomness (0-1, note: different range than OpenAI!)
            - 0: Most deterministic
            - 1: Maximum creativity
        system_prompt (str, optional): Defines Claude's behavior
            Note: In Anthropic API, this is a separate parameter, not a message
    
    Returns:
        dict: {
            'text': Response content,
            'model': Actual model used,
            'input_tokens': Tokens in prompt,
            'output_tokens': Tokens in response,
            'total_tokens': Total tokens used,
            'input_cost': Cost of input in USD,
            'output_cost': Cost of output in USD,
            'total_cost': Total cost in USD
        }
        OR {'error': error_message} if the call fails
    
    Medical AI Note:
        The detailed token/cost information is valuable for:
        - Audit trails in clinical applications
        - Cost allocation per department/project
        - Optimizing system prompts for efficiency
    """
    # Fallback for demonstration when API key is not configured
    if not ANTHROPIC_AVAILABLE:
        return {
            "text": f"[SIMULATED RESPONSE]\nThis is a simulated response for: {prompt[:100]}...",
            "model": model,
            "input_tokens": 0,
            "output_tokens": 0,
            "total_tokens": 0,
            "input_cost": 0.0,
            "output_cost": 0.0,
            "total_cost": 0.0
        }
    
    try:
        # Import and initialize Anthropic client
        # Client automatically uses ANTHROPIC_API_KEY from environment
        from anthropic import Anthropic
        client = Anthropic()
        
        # Build request parameters
        # Note: Anthropic uses kwargs pattern for flexibility
        kwargs = {
            "model": model,
            "max_tokens": 1024,          # Maximum response length
            "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}]
        }
        
        # Add system prompt if provided (separate from messages in Anthropic API)
        if system_prompt:
            kwargs["system"] = system_prompt
        
        # Make the API call
        response = client.messages.create(**kwargs)
        
        # Extract token usage from response
        # Unlike OpenAI, Anthropic includes this in every response
        input_tokens = response.usage.input_tokens
        output_tokens = response.usage.output_tokens
        total_tokens = input_tokens + output_tokens
        
        # Calculate estimated cost
        cost_info = calculate_cost(response.model, input_tokens, output_tokens)
        
        # Return comprehensive response dictionary
        return {
            "text": response.content[0].text,  # Note: .text not .content
            "model": response.model,            # Actual model used (may differ from requested)
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
            "input_cost": cost_info["input_cost"],
            "output_cost": cost_info["output_cost"],
            "total_cost": cost_info["total_cost"]
        }
        
    except Exception as e:
        # Return error in same dict format for consistent handling
        return {"error": f"Error in API call: {e}"}


# =============================================================================
# EXAMPLE: Medical Question with Cost Tracking
# =============================================================================

result = chat_with_claude(
    prompt="What is the difference between Type 1 and Type 2 diabetes?",
    temperature=0.3,  # Low temperature for factual medical information
    system_prompt="You are a medical assistant. Explain in a patient-friendly way."
)

print("=" * 60)
print("CLAUDE API EXAMPLE: Medical Question with Cost Tracking")
print("=" * 60)

if "error" in result:
    print(f"\n❌ Error: {result['error']}")
else:
    print(f"\n📊 API Usage Statistics:")
    print(f"   Model: {result['model']}")
    print(f"   Input tokens:  {result['input_tokens']:,}")
    print(f"   Output tokens: {result['output_tokens']:,}")
    print(f"   Total tokens:  {result['total_tokens']:,}")
    print(f"\n💰 Cost Breakdown:")
    print(f"   Input cost:  ${result['input_cost']:.6f}")
    print(f"   Output cost: ${result['output_cost']:.6f}")
    print(f"   Total cost:  ${result['total_cost']:.6f} ({result['total_cost']*100:.4f} cents)")
    print(f"\n📝 Response:\n{result['text']}")


CLAUDE API EXAMPLE: Medical Question with Cost Tracking

📊 API Usage Statistics:
   Model: claude-sonnet-4-5-20250929
   Input tokens:  38
   Output tokens: 319
   Total tokens:  357

💰 Cost Breakdown:
   Input cost:  $0.000114
   Output cost: $0.004785
   Total cost:  $0.004899 (0.4899 cents)

📝 Response:
# Type 1 vs Type 2 Diabetes

Think of insulin as a "key" that unlocks your cells so sugar (glucose) from food can get inside and give you energy.

## **Type 1 Diabetes**
- **What happens**: Your body stops making insulin (no keys at all)
- **Why**: Your immune system mistakenly attacks the cells that make insulin
- **Who gets it**: Usually starts in children and young adults, but can happen at any age
- **Treatment**: Must take insulin every day (by injection or pump) to survive
- **Prevention**: Can't be prevented

## **Type 2 Diabetes**
- **What happens**: Your body either doesn't make enough insulin OR your cells don't respond to it well (the keys don't work properly)
- **Why**: U

### ✅ Check Your Understanding: Claude API & Token Economics

**Q1:** Why is output typically more expensive than input tokens (e.g., $15 vs $3 per million for Sonnet)?

<details>
<summary>Click for answer</summary>

**Answer:** Output generation is computationally more expensive than input processing because:
- **Generation is sequential**: Each output token depends on all previous tokens, requiring many forward passes
- **Input is parallel**: The entire prompt can be processed in one pass
- **Longer inference time**: Each output token adds latency and compute cost
- **Business model**: Providers charge more for the value they deliver (the response)

This pricing structure encourages **concise prompts** and **appropriate response length limits**.
</details>

**Q2:** A medical chatbot handles 10,000 queries per day, averaging 200 input tokens and 400 output tokens per query. Using Claude Sonnet, what's the estimated daily cost?

<details>
<summary>Click for answer</summary>

**Answer:** 
- **Daily input tokens**: 10,000 × 200 = 2,000,000 (2M)
- **Daily output tokens**: 10,000 × 400 = 4,000,000 (4M)
- **Input cost**: 2M ÷ 1M × $3.00 = $6.00
- **Output cost**: 4M ÷ 1M × $15.00 = $60.00
- **Total daily cost**: $6.00 + $60.00 = **$66.00/day** (~$2,000/month)

This demonstrates why **output costs typically dominate** API expenses!
</details>

**Q3:** What's the key difference in how OpenAI and Anthropic handle system prompts in their APIs?

<details>
<summary>Click for answer</summary>

**Answer:** 
- **OpenAI**: System prompt is included as a message in the `messages` array with `role: "system"`
- **Anthropic**: System prompt is a separate top-level parameter (`system=...`)

Both achieve the same goal of defining model behavior, but the API structure differs. This matters when building abstraction layers that support multiple providers.
</details>

**Q4:** When would you choose Claude Haiku over Claude Opus for a medical application?

<details>
<summary>Click for answer</summary>

**Answer:** Choose **Haiku** (fast, cheap) for:
- Simple triage questions with clear answers
- High-volume, low-complexity tasks
- Real-time applications requiring low latency
- Initial screening before escalating to more capable models

Choose **Opus** (capable, expensive) for:
- Complex medical reasoning requiring nuance
- Analyzing lengthy patient histories
- Tasks requiring strong accuracy where cost is secondary
- Research applications needing best-in-class performance
</details>

---

## 4. Error Handling

When using APIs programmatically, robust error handling is **critical**, especially in healthcare contexts where system failures could impact patient care.

### Common API Errors and Their Causes

| Error Type | HTTP Code | Cause | Solution |
|------------|-----------|-------|----------|
| **Rate Limit** | 429 | Too many requests | Implement backoff, queue requests |
| **Authentication** | 401 | Invalid API key | Check key validity, rotation |
| **Bad Request** | 400 | Malformed request | Validate input before sending |
| **Server Error** | 500/503 | Provider issues | Retry with backoff |
| **Timeout** | - | Network/long response | Set appropriate timeouts |
| **Context Length** | 400 | Input too long | Truncate or chunk input |

### Error Handling Strategy: Exponential Backoff

When an API is temporarily unavailable (e.g., rate limiting), we use **exponential backoff**:

```
Attempt 1: Wait 1 second
Attempt 2: Wait 2 seconds
Attempt 3: Wait 4 seconds
...
```

This prevents overwhelming the server while eventually succeeding.

> **Medical AI Note**: In clinical systems, consider having a **fallback mechanism** (e.g., cached responses, human escalation) when API calls fail repeatedly.

In [4]:
# =============================================================================
# ROBUST API CLIENT WITH RETRY LOGIC
# =============================================================================
# This class implements production-grade error handling patterns:
#
# 1. RATE LIMITING: Prevents overwhelming the API with too many requests
#    - Enforces minimum time between requests
#    - Essential for staying within API quotas
#
# 2. EXPONENTIAL BACKOFF: When errors occur, wait progressively longer
#    - Attempt 1: Wait 1s, Attempt 2: Wait 2s, Attempt 3: Wait 4s
#    - Gives temporary issues time to resolve
#
# 3. RETRY LOGIC: Automatically retry failed requests
#    - Handles transient network issues
#    - Distinguishes retryable vs fatal errors
#
# Medical AI Context:
#    In healthcare systems, reliability is paramount. A robust client
#    ensures the system degrades gracefully under load or when APIs
#    have temporary issues, rather than failing completely.
# =============================================================================

import time
from typing import Callable, Any


class RobustAPIClient:
    """
    A production-ready API client wrapper with built-in resilience.
    
    This class wraps API calls to provide:
    - Automatic retry on transient failures
    - Exponential backoff for rate limiting
    - Request throttling to respect API limits
    
    Example usage:
        client = RobustAPIClient(max_retries=3, base_delay=1.0)
        result = client.call_api(chat_with_gpt, "What is diabetes?")
    
    Attributes:
        max_retries (int): Maximum number of retry attempts
        base_delay (float): Initial delay in seconds for exponential backoff
        last_call_time (float): Timestamp of last API call
        min_interval (float): Minimum seconds between API calls
    """
    
    def __init__(self, max_retries: int = 3, base_delay: float = 1.0):
        """
        Initialize the robust API client.
        
        Args:
            max_retries: How many times to retry failed requests (default: 3)
            base_delay: Starting delay for exponential backoff in seconds (default: 1.0)
        """
        self.max_retries = max_retries
        self.base_delay = base_delay
        self.last_call_time = 0
        self.min_interval = 0.5  # Minimum 500ms between calls
    
    def _wait_if_needed(self) -> None:
        """
        Implement rate limiting by enforcing minimum interval between calls.
        
        This prevents overwhelming the API with rapid-fire requests,
        which could trigger rate limiting or cause service degradation.
        """
        elapsed = time.time() - self.last_call_time
        if elapsed < self.min_interval:
            sleep_time = self.min_interval - elapsed
            time.sleep(sleep_time)
        self.last_call_time = time.time()
    
    def call_api(self, func: Callable, *args, **kwargs) -> Any:
        """
        Execute an API call with automatic retry and backoff.
        
        This method wraps any function (typically an API call) and handles:
        - Transient failures with automatic retry
        - Rate limiting with exponential backoff
        - Non-retryable errors with immediate return
        
        Args:
            func: The function to call (e.g., chat_with_gpt)
            *args: Positional arguments to pass to func
            **kwargs: Keyword arguments to pass to func
        
        Returns:
            The result of func(*args, **kwargs) if successful,
            or an error message string if all retries fail.
        
        Example:
            >>> client = RobustAPIClient()
            >>> result = client.call_api(
            ...     chat_with_gpt,
            ...     "Explain hypertension",
            ...     temperature=0.3
            ... )
        """
        for attempt in range(self.max_retries):
            try:
                # Respect rate limits before making call
                self._wait_if_needed()
                
                # Execute the actual API call
                return func(*args, **kwargs)
                
            except Exception as e:
                error_msg = str(e).lower()
                
                # Check for rate limit errors (HTTP 429 or explicit message)
                if "rate_limit" in error_msg or "429" in str(e):
                    # Calculate exponential backoff delay
                    # 2^attempt: 1s, 2s, 4s, 8s, etc.
                    wait_time = self.base_delay * (2 ** attempt)
                    print(f"⏳ Rate limit hit - waiting {wait_time}s "
                          f"(retry {attempt + 1}/{self.max_retries})")
                    time.sleep(wait_time)
                    # Continue to next retry attempt
                else:
                    # Non-retryable error (e.g., authentication, bad request)
                    # Return immediately rather than wasting retries
                    return f"❌ Error after {attempt + 1} attempts: {e}"
        
        # All retries exhausted
        return "❌ Maximum retry attempts reached - API unavailable"


# =============================================================================
# DEMONSTRATION
# =============================================================================
print("=" * 60)
print("RobustAPIClient initialized")
print("=" * 60)
print("\n📋 Configuration:")
print("   Max retries: 3")
print("   Base delay: 1.0s (exponential backoff: 1s → 2s → 4s)")
print("   Min interval: 0.5s between requests")
print("\n✓ Ready for production use")
print("\n💡 Usage example:")
print("   client = RobustAPIClient(max_retries=5)")
print("   result = client.call_api(chat_with_gpt, 'Your prompt here')")

RobustAPIClient initialized

📋 Configuration:
   Max retries: 3
   Base delay: 1.0s (exponential backoff: 1s → 2s → 4s)
   Min interval: 0.5s between requests

✓ Ready for production use

💡 Usage example:
   client = RobustAPIClient(max_retries=5)
   result = client.call_api(chat_with_gpt, 'Your prompt here')


### ✅ Check Your Understanding: Error Handling

**Q1:** What is "exponential backoff" and why is it used instead of fixed delays?

<details>
<summary>Click for answer</summary>

**Answer:** Exponential backoff progressively increases wait time between retries (1s → 2s → 4s → 8s...). Benefits over fixed delays:
- **Prevents thundering herd**: If many clients retry simultaneously, exponential backoff spreads them out
- **Allows recovery time**: Gives overloaded systems progressively more time to recover
- **Efficient resource use**: Avoids wasting resources on rapid retries when the issue persists
- **Fair to other users**: Reduces load on shared infrastructure
</details>

**Q2:** A clinical decision support system fails to get an API response. What should happen next?

<details>
<summary>Click for answer</summary>

**Answer:** A well-designed medical system should have multiple fallback layers:

1. **Retry with backoff** (handled by RobustAPIClient)
2. **Try alternative model/provider** (e.g., switch from GPT-4 to Claude)
3. **Use cached response** for common queries
4. **Degrade gracefully** by showing partial information
5. **Alert a human** if the query is critical
6. **Log the failure** for later analysis
7. **Never silently fail** in ways that could harm patients
</details>

**Q3:** Why does the `RobustAPIClient` implement `min_interval` between calls even when no error occurs?

<details>
<summary>Click for answer</summary>

**Answer:** Proactive rate limiting prevents problems:
- **Stays under quota**: API providers often have requests-per-minute limits
- **Avoids triggering 429s**: Prevention is better than handling errors
- **Smoother operation**: Consistent pacing prevents bursts and dips
- **Considerate usage**: Shared infrastructure benefits from steady load
- **Cost control**: Prevents accidental spending spikes from loops
</details>

---

## 5. Practical Examples

### Building a Medical Assistant with Safety Constraints

When building LLM-based medical tools, **safety by design** is essential. This means embedding safety constraints directly into the system prompt rather than relying on users or downstream filtering.

#### Key Safety Patterns for Medical LLMs

| Pattern | Implementation | Purpose |
|---------|----------------|---------|
| **Scope limitation** | "Only answer questions about X" | Prevents off-topic advice |
| **Disclaimer injection** | "Always include..." | Legal/ethical protection |
| **Uncertainty expression** | "Express confidence levels" | Prevents overconfidence |
| **Referral triggers** | "For symptoms X, Y, Z recommend..." | Routes to appropriate care |
| **Knowledge boundaries** | "If uncertain, say so" | Prevents hallucination harm |

### Example: Medical Assistant Function

In [5]:
# =============================================================================
# MEDICAL ASSISTANT WITH EMBEDDED SAFETY CONSTRAINTS
# =============================================================================
# This function demonstrates how to build an LLM-based medical assistant
# with appropriate safety guardrails. Key design decisions:
#
# 1. SYSTEM PROMPT: Defines strict behavioral boundaries
# 2. CONTEXT HANDLING: Allows relevant medical history
# 3. EXPLICIT WARNINGS: Every response includes disclaimers
# 4. STRUCTURED OUTPUT: Returns both answer and metadata
#
# In production, this would:
# - Log all queries and responses for audit
# - Include user authentication
# - Implement input validation/sanitization
# - Connect to approved medical knowledge bases
# =============================================================================


def medical_assistant(question: str, context: str = None) -> dict:
    """
    A safety-constrained medical information assistant.
    
    This function wraps LLM API calls with appropriate guardrails
    for providing general health information while maintaining
    clear boundaries about what it can and cannot do.
    
    Args:
        question (str): The user's health-related question
        context (str, optional): Relevant medical context
            Example: "Patient with newly diagnosed type 2 diabetes"
    
    Returns:
        dict: {
            'answer': The assistant's response with embedded disclaimers,
            'warnings': List of important warnings to display to user
        }
    
    Safety Features:
        - System prompt enforces general information only
        - Explicit prohibition on diagnosis/treatment recommendations
        - Automatic referral to healthcare professionals
        - Emergency symptom detection with urgent care advice
        
    Example:
        >>> result = medical_assistant(
        ...     "What is the normal range for blood pressure?",
        ...     context="50-year-old with family history of hypertension"
        ... )
        >>> print(result['answer'])
    """
    
    # -------------------------------------------------------------------------
    # SAFETY-FIRST SYSTEM PROMPT
    # -------------------------------------------------------------------------
    # This prompt embeds critical safety constraints directly into the model's
    # behavior. Note the specific, actionable rules rather than vague guidelines.
    
    system_prompt = """You are a medical INFORMATION assistant (not a medical advisor).

CRITICAL SAFETY RULES - NEVER VIOLATE THESE:

1. SCOPE: Provide ONLY general health information from established medical sources
   - You explain concepts, not recommend actions
   - You describe symptoms, not diagnose conditions

2. NO DIAGNOSIS: Never suggest what condition a patient might have
   - Wrong: "This sounds like you might have X"
   - Right: "Symptom X can be associated with various conditions including..."

3. NO TREATMENT: Never recommend specific treatments or medications
   - Wrong: "You should take medication Y"
   - Right: "Treatment options a doctor might consider include..."

4. ALWAYS REFER: End every response by recommending professional consultation
   - "For your specific situation, please consult your healthcare provider"

5. UNCERTAINTY: If you're not sure, say so explicitly
   - "I don't have reliable information about this"
   - "This is outside my knowledge - please consult a specialist"

6. EMERGENCIES: For these symptoms, IMMEDIATELY recommend emergency care:
   - Chest pain, difficulty breathing, signs of stroke
   - Severe bleeding, loss of consciousness
   - Thoughts of self-harm or harming others
   - Say: "Please seek immediate medical attention (call emergency services)"

Answer in English. Be accurate, helpful, and appropriately cautious."""
    
    # -------------------------------------------------------------------------
    # CONTEXT INTEGRATION
    # -------------------------------------------------------------------------
    # Context allows more relevant responses while maintaining safety
    
    full_prompt = question
    if context:
        # Prepend context to help model give relevant information
        full_prompt = f"Context: {context}\n\nQuestion: {question}"
    
    # -------------------------------------------------------------------------
    # API CALL (or simulation)
    # -------------------------------------------------------------------------
    # In production, this would call the actual API
    # For this notebook, we demonstrate with simulated response
    
    # Production code would be:
    # response = chat_with_gpt(
    #     prompt=full_prompt,
    #     system_prompt=system_prompt,
    #     temperature=0.3  # Low temperature for factual responses
    # )
    
    # Simulated response for demonstration
    simulated_answer = {
        "answer": (
            f"[Simulated response for: {question[:50]}...]\n\n"
            "This would contain general health information about your question, "
            "explained in accessible terms.\n\n"
            "**Important:** This is general information only. For advice about "
            "your specific health situation, please consult your healthcare provider."
        ),
        "warnings": [
            "This information does not replace professional medical advice",
            "Consult a healthcare provider for personal medical assessment",
            "In case of emergency, call emergency services immediately"
        ]
    }
    
    return simulated_answer


# =============================================================================
# DEMONSTRATION
# =============================================================================

result = medical_assistant(
    question="What are common side effects of metformin?",
    context="Patient with newly diagnosed type 2 diabetes"
)

print("=" * 60)
print("MEDICAL ASSISTANT DEMONSTRATION")
print("=" * 60)
print(f"\n📋 Question: What are common side effects of metformin?")
print(f"📄 Context: Patient with newly diagnosed type 2 diabetes")
print(f"\n{'─' * 60}")
print(f"\n📝 Response:\n{result['answer']}")
print(f"\n{'─' * 60}")
print(f"\n⚠️  Warnings:")
for warning in result['warnings']:
    print(f"   • {warning}")

MEDICAL ASSISTANT DEMONSTRATION

📋 Question: What are common side effects of metformin?
📄 Context: Patient with newly diagnosed type 2 diabetes

────────────────────────────────────────────────────────────

📝 Response:
[Simulated response for: What are common side effects of metformin?...]

This would contain general health information about your question, explained in accessible terms.

**Important:** This is general information only. For advice about your specific health situation, please consult your healthcare provider.

────────────────────────────────────────────────────────────

⚠️  Warnings:
   • This information does not replace professional medical advice
   • Consult a healthcare provider for personal medical assessment
   • In case of emergency, call emergency services immediately


---

## 6. Security Considerations

When deploying LLM APIs in healthcare contexts, security and compliance are **non-negotiable**. This section covers the key concerns and mitigation strategies.

### 6.1 Privacy and Data Protection

| Concern | Risk | Mitigation |
|---------|------|------------|
| **Data transmission** | Patient data sent to external servers | Anonymize/pseudonymize before API calls |
| **Data retention** | API providers may store prompts | Review provider data policies (e.g., OpenAI's data usage policy) |
| **Regulatory compliance** | GDPR, HIPAA, local health data laws | Use compliant providers or local models |
| **Data minimization** | Sending more data than needed | Include only essential information in prompts |

> **Recommendation**: For truly sensitive data, consider **local models** (Ollama, llama.cpp) that never leave your infrastructure. See our Lab3 materials on local deployment.

### 6.2 Hallucination and Accuracy

LLMs can confidently generate incorrect information—a critical risk in healthcare.

| Strategy | Implementation |
|----------|----------------|
| **Low temperature** | Use `temperature=0-0.3` for factual queries |
| **Verification layer** | Cross-reference with approved medical databases |
| **Source citation** | Ask model to cite sources (then verify them!) |
| **Uncertainty expression** | Train users to recognize hedging language |
| **Human review** | All clinical outputs reviewed by qualified personnel |

### 6.3 Audit Logging

Comprehensive logging is essential for:
- **Traceability**: Who asked what, when?
- **Quality assurance**: Reviewing AI outputs over time
- **Incident investigation**: Understanding failures
- **Compliance**: Demonstrating appropriate use

```python
# Example log entry structure (DO NOT log actual patient data!)
log_entry = {
    "timestamp": "2026-01-19T14:30:00Z",
    "user_id": "anonymized_hash",
    "query_type": "medication_info",
    "model": "claude-sonnet-4-5",
    "tokens_used": 450,
    "response_reviewed": True,
    "reviewer_id": "clinician_hash"
}
```

### 6.4 Human-in-the-Loop

AI in healthcare should **augment**, not replace, clinical judgment.

```
┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│   Patient   │────▶│   AI Tool   │────▶│  Clinician  │────▶ Decision
│   Query     │     │  (Suggests) │     │  (Decides)  │
└─────────────┘     └─────────────┘     └─────────────┘
                           │
                           ▼
                    ┌─────────────┐
                    │   Logging   │
                    │   & Audit   │
                    └─────────────┘
```

### 6.5 Security Checklist

Before deploying an LLM-based medical tool, verify:

- [ ] API keys stored securely (not in code, not in git)
- [ ] Data anonymization/pseudonymization implemented
- [ ] Provider data processing agreement reviewed
- [ ] Compliance with relevant regulations confirmed
- [ ] Audit logging implemented
- [ ] Human review workflow established
- [ ] Error handling and fallbacks tested
- [ ] User training on AI limitations completed
- [ ] Incident response plan documented

### ✅ Check Your Understanding: Security & Ethics

**Q1:** A hospital wants to use ChatGPT to help doctors write discharge summaries. What data protection measures should be implemented?

<details>
<summary>Click for answer</summary>

**Answer:** Essential data protection measures:
1. **Anonymization**: Remove all patient identifiers (name, ID, dates) before sending to API
2. **Data agreement**: Review OpenAI's data processing agreement for healthcare compliance
3. **Consider alternatives**: Evaluate local models that don't transmit data externally
4. **Minimal data**: Send only information essential for the summary
5. **Audit trail**: Log all API interactions (without patient data)
6. **Clinician review**: All generated content reviewed before use
7. **Patient consent**: Inform patients about AI use in their care documentation
</details>

**Q2:** Why is "human-in-the-loop" considered essential for medical AI, not just recommended?

<details>
<summary>Click for answer</summary>

**Answer:** Human oversight is essential because:
- **Liability**: Ultimate responsibility lies with licensed professionals
- **Context**: AI lacks access to full patient context, history, and non-verbal cues
- **Edge cases**: Unusual presentations may confuse AI
- **Hallucinations**: AI can generate plausible but incorrect information
- **Ethical judgment**: Treatment decisions involve values, preferences, and ethics
- **Regulatory**: Most jurisdictions require human decision-making for medical care
- **Trust**: Patients expect human judgment in healthcare decisions
</details>

**Q3:** A medical chatbot gives incorrect medication dosage information. Who is responsible?

<details>
<summary>Click for answer</summary>

**Answer:** Responsibility is complex and typically shared:
- **Development team**: For inadequate safety guardrails and testing
- **Deploying organization**: For insufficient validation and oversight processes
- **Reviewing clinician**: If they approved output without verification
- **Regulatory body**: For approving (or not regulating) the tool

This is why:
- AI tools should clearly disclaim that they don't provide medical advice
- Human verification should be mandatory for clinical use
- Errors should trigger incident reports and system improvements
- Liability should be clearly defined before deployment
</details>

**Q4:** What is "data minimization" and why does it matter for LLM API calls?

<details>
<summary>Click for answer</summary>

**Answer:** Data minimization means sending only the **minimum necessary information** to accomplish a task.

For LLM APIs, this matters because:
- **Privacy**: Less data transmitted = less exposure risk
- **Compliance**: GDPR and other regulations require data minimization
- **Cost**: Fewer tokens = lower API costs
- **Security**: Smaller attack surface if data is intercepted
- **Performance**: Shorter prompts may get better, more focused responses

**Example**: Instead of sending a full patient record, send only: "65-year-old with hypertension asking about low-sodium diet options"
</details>

---

<a id="summary"></a>
## 🎯 Key Takeaways

### The Big Picture

```
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                    LLM API Integration for Medical Applications                     │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│   Secure          API           Robust         Safety           Human               │
│   Config    →     Calls    →    Handling   →   Prompts    →    Review               │
│   (.env)          (GPT/Claude)  (Retry/Backoff) (Constraints)   (Always!)           │
│                                                                                     │
│   Foundation      Execution     Resilience     Guardrails      Oversight            │
└─────────────────────────────────────────────────────────────────────────────────────┘
```

### Core Concepts Summary

| # | Concept | Key Insight | Remember This |
|---|---------|-------------|---------------|
| 1 | **Secure API keys** | Never hardcode, use environment variables | `.env` files + `.gitignore` |
| 2 | **Temperature control** | 0 = deterministic, 1+ = creative | Use low temp for medical facts |
| 3 | **System prompts** | Define model behavior and constraints | Embed safety rules directly |
| 4 | **Token economics** | Input cheap, output expensive | Monitor costs, optimize prompts |
| 5 | **Error handling** | Exponential backoff for resilience | `RobustAPIClient` pattern |
| 6 | **Rate limiting** | Proactive throttling prevents errors | Minimum interval between calls |
| 7 | **Data privacy** | Anonymize before sending to APIs | Consider local models for PHI |
| 8 | **Human oversight** | AI augments, doesn't replace | Always clinician-reviewed |

### OpenAI vs Anthropic: Quick Comparison

| Aspect | OpenAI (ChatGPT) | Anthropic (Claude) |
|--------|------------------|-------------------|
| **System prompt** | Message in array | Separate parameter |
| **Temperature** | 0-2 | 0-1 |
| **Token info** | Separate request | In every response |
| **Pricing model** | Per-token | Per-token |
| **Best for** | Wide model selection | Safety-focused apps |

### What You Can Now Implement

After completing this notebook, you should be able to:

- ✅ Set up secure API key management for development and production
- ✅ Make API calls to both OpenAI and Anthropic with appropriate parameters
- ✅ Implement robust error handling with retry logic
- ✅ Build medical-context LLM functions with safety constraints
- ✅ Track and estimate API costs for budgeting
- ✅ Apply privacy and security best practices
- ✅ Design human-in-the-loop workflows for clinical AI

### Production Readiness Checklist

Before deploying an LLM-based medical tool:

| Category | Requirements |
|----------|--------------|
| **Security** | ☐ API keys secured ☐ Data anonymization ☐ Audit logging |
| **Reliability** | ☐ Error handling ☐ Rate limiting ☐ Fallback mechanisms |
| **Safety** | ☐ Safety prompts ☐ Output validation ☐ Human review workflow |
| **Compliance** | ☐ Privacy policy ☐ Data agreements ☐ Regulatory approval |
| **Operations** | ☐ Cost monitoring ☐ Performance metrics ☐ Incident response |

---

<a id="glossary-of-key-terms"></a>
## 📚 Glossary of Key Terms

| Term | Definition | Medical Context |
|------|------------|-----------------|
| **API (Application Programming Interface)** | A set of protocols for building and integrating application software | How clinical systems communicate with LLM services |
| **API Key** | Secret token for authenticating API requests | Must be secured like patient credentials |
| **Backoff (Exponential)** | Strategy of progressively increasing wait times between retries | Ensures clinical system resilience during outages |
| **Chat Completion** | API endpoint for conversational AI interactions | Primary interface for medical chatbots |
| **Context Window** | Maximum tokens a model can process at once | Limits how much patient history can be included |
| **Environment Variable** | Dynamic value set outside the application code | Secure method for API key storage |
| **Hallucination** | When AI generates plausible but false information | Critical risk in medical information systems |
| **Human-in-the-Loop** | Design requiring human oversight of AI decisions | Essential for clinical AI applications |
| **Input Tokens** | Tokens in the prompt sent to the model | Affects cost and context usage |
| **Max Tokens** | Limit on response length | Controls output verbosity and cost |
| **Output Tokens** | Tokens in the model's response | Primary cost driver for API usage |
| **PHI (Protected Health Information)** | Individually identifiable health information | Must never be sent to external APIs without safeguards |
| **Prompt** | Text input to an LLM | The question or instruction sent to the model |
| **Rate Limiting** | Restricting the number of API requests over time | Prevents quota exhaustion and 429 errors |
| **Retry Logic** | Automatically re-attempting failed operations | Critical for production reliability |
| **System Prompt** | Instructions defining model behavior and constraints | Where safety guardrails are embedded |
| **Temperature** | Parameter controlling output randomness (0-1 or 0-2) | Low for facts, higher for creative content |
| **Token** | Basic unit of text processing (~4 characters or 0.75 words) | Basis for API pricing and limits |
| **Wrapper Function** | Code that simplifies or enhances another function | `chat_with_gpt()` wraps the OpenAI API |

---

<a id="further-reading--resources"></a>
## 📖 Further Reading & Resources

### Official Documentation

| Resource | Description | Link |
|----------|-------------|------|
| **OpenAI API Docs** | Complete reference for ChatGPT API | [platform.openai.com/docs](https://platform.openai.com/docs) |
| **Anthropic API Docs** | Complete reference for Claude API | [docs.anthropic.com](https://docs.anthropic.com) |
| **OpenAI Cookbook** | Practical examples and best practices | [cookbook.openai.com](https://cookbook.openai.com/) |
| **Anthropic Cookbook** | Claude-specific examples | [github.com/anthropics/anthropic-cookbook](https://github.com/anthropics/anthropic-cookbook) |

### Medical AI & Healthcare LLMs

| Resource | Description |
|----------|-------------|
| **AMIA (American Medical Informatics Association)** | Professional organization for health informatics |
| **FDA Guidance on AI/ML in Medical Devices** | Regulatory framework for medical AI |
| **WHO Ethics of AI in Health** | International ethical guidelines |
| **Nature Digital Medicine** | Journal covering digital health and AI |

### Security & Privacy

| Resource | Description |
|----------|-------------|
| **HIPAA Compliance** | US healthcare data protection requirements |
| **GDPR** | EU data protection regulation |
| **OpenAI Data Usage Policies** | How OpenAI handles API data |
| **Anthropic Privacy Policy** | How Anthropic handles API data |

### Technical Deep Dives

| Topic | Recommended Resource |
|-------|----------------------|
| **Prompt Engineering** | Anthropic's prompt engineering guide |
| **Token Counting** | tiktoken library (OpenAI) for estimating tokens |
| **Local LLMs** | Ollama, llama.cpp for private deployments |
| **LLM Evaluation** | HELM benchmark, medical QA datasets |

### Related Lab Notebooks

| Notebook | Topics |
|----------|--------|
| `01-introduction-genai.ipynb` | GenAI fundamentals |
| `04-prompt-engineering.ipynb` | Prompt design techniques |
| `06-ai-ethics-medicine.ipynb` | Ethics in medical AI |
| `07-trustworthy-ai.ipynb` | Building trustworthy AI systems |

---

## 🙏 Acknowledgments

This notebook was developed for BMED365 at the University of Bergen. It combines practical API skills with healthcare-specific considerations for medical AI applications.

---

*Back to [Lab 3 overview](../README.md)*

---

*Last updated: 2026-01-19*